In [1]:
import numpy as np
import pandas as pd




# Get data

In [2]:
hist = pd.read_csv("/Users/macbookpro/platform/Backend/data/processed/default_hist.csv")
orig = pd.read_csv('/Users/macbookpro/platform/Backend/data/raw/orig_data_col.csv')



/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_78696/2619987728.py:2: DtypeWarning: Columns (24,25,29) have mixed types. Specify dtype option on import or set low_memory=False.
  orig = pd.read_csv('/Users/macbookpro/platform/Backend/data/raw/orig_data_col.csv')


In [3]:
hist.columns

Index(['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD',
       'CURRENT_ACTUAL_UPB', 'CURRENT_LOAN_DELINQUENCY_STATUS', 'LOAN_AGE',
       'REMAINING_MONTHS_TO_LEGAL_MATURITY', 'DEFECT_SETTLEMENT_DATE',
       'MODIFICATION_FLAG', 'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE',
       'CURRENT_INTEREST_RATE', 'CURRENT_NON_INTEREST_BEARING_UPB',
       'DUE_DATE_OF_LAST_PAID_INSTALLMENT', 'MI_RECOVERIES',
       'NET_SALE_PROCEEDS', 'NON_MI_RECOVERIES', 'TOTAL_EXPENSES',
       'LEGAL_COSTS', 'MAINTENANCE_AND_PRESERVATION_COSTS',
       'TAXES_AND_INSURANCE', 'MISCELLANEOUS_EXPENSES',
       'ACTUAL_LOSS_CALCULATION', 'CUMULATIVE_MODIFICATION_COST',
       'INTEREST_RATE_STEP_INDICATOR', 'PAYMENT_DEFERRAL_FLAG',
       'ESTIMATED_LTV', 'ZERO_BALANCE_REMOVAL_UPB',
       'DELINQUENT_ACCRUED_INTEREST', 'DELINQUENCY_DUE_TO_DISASTER',
       'BORROWER_ASSISTANCE_STATUS_CODE', 'CURRENT_MONTH_MODIFICATION_COST',
       'INTEREST_BEARING_UPB'],
      dtype='object')

In [34]:
orig.columns

Index(['CREDIT_SCORE', 'FIRST_TIME_HOMEBUYER_FLAG', 'MSA', 'MI_PERCENTAGE',
       'NUMBER_OF_UNITS', 'OCCUPANCY_STATUS', 'OCLTV', 'DTI', 'ORIGINAL_UPB',
       'LTV', 'ORIGINAL_INTEREST_RATE', 'CHANNEL', 'PPM_FLAG', 'PRODUCT_TYPE',
       'STATE', 'PROPERTY_TYPE', 'POSTAL_CODE', 'LOAN_SEQUENCE_NUMBER',
       'LOAN_PURPOSE', 'ORIGINAL_LOAN_TERM', 'NUMBER_OF_BORROWERS',
       'SELLER_NAME', 'SERVICER_NAME', 'SUPER_CONFORMING_FLAG',
       'PRE_RELIEF_REFI_LOAN_SEQ', 'PROGRAM_INDICATOR',
       'RELIEF_REFINANCE_INDICATOR', 'PROPERTY_VALUATION_METHOD', 'IO_FLAG',
       'MORTGAGE_INSURANCE_CANCELLATION', 'IS_MISSING_CREDIT_SCORE',
       'IS_MISSING_DTI'],
      dtype='object')

In [4]:
loan_hist = hist[hist['LOAN_SEQUENCE_NUMBER'] == 'F07Q10000071']
loan_orig = orig[orig['LOAN_SEQUENCE_NUMBER'] == 'F07Q10000071']


# Config

In [19]:
train_cfPath = '/Users/macbookpro/platform/Backend/configs/ressources/Lgd_class_train.yaml'
test_cfPath = '/Users/macbookpro/platform/Backend/configs/ressources/Lgd_class_test.yaml'

# Feature and scaler

In [5]:
import importlib
import src.LGDcomponent.pipelines.lgdFeaturePipeline as lgdFeaturePipeline
importlib.reload(lgdFeaturePipeline)
from src.LGDcomponent.pipelines.lgdFeaturePipeline import LGDFeaturePipeline

/Users/macbookpro/platform/Backend/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
pipeline = LGDFeaturePipeline(config_path=train_cfPath)

scaler config loaded successfully


In [7]:
X,y = pipeline.build(hist, orig)

Copy DataFrame     : 1.5s
Cast DPD           : 0.1s
Groupby            : 0.0s
Colonnes de travail: 0.7s
[LGD] Loans exclus pour EAD=0 (artefact de séquence) : 196
[LGD] Loans avec target calculée : 30807 / 30807
[LGD] Observations clippées hors [0,1] : 4238
[LGD] Distribution LGD :
count    30807.0000
mean         0.3668
std          0.2918
min          0.0000
25%          0.1260
50%          0.3288
75%          0.5417
max          1.0000
Name: lgd_target, dtype: float64
🏃 View run LGD_scaler_preprocessing_v1 at: http://localhost:5000/#/experiments/3/runs/239eb675b01c481eacf5837cc911a393
🧪 View experiment at: http://localhost:5000/#/experiments/3
datas scaled


In [8]:
X.shape

(30807, 20)

In [9]:
y.shape

(30807,)

# Split and save

In [10]:
from sklearn.model_selection import train_test_split

# 1. Separate the full dataset into Training (80%) and a temporary Test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 2. Split the temp set into Training (70%) and Validation (30% of the temp set, or 15% of the total)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.30, random_state=42)

In [14]:
#X_train.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_train.csv',sep=',',index=False)
#X_val.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_val.csv',sep=',',index=False)
#X_test.to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/x_test.csv',sep=',',index=False)

In [15]:
#pd.DataFrame(y_train).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_train.csv',sep=',',index=False)
#pd.DataFrame(y_val).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_val.csv',sep=',',index=False)
#pd.DataFrame(y_test).to_csv('/Users/macbookpro/platform/Backend/data/processed/LGD/y_test.csv',sep=',',index=False)#

# Run

train

In [11]:

import src.LGDcomponent.run.lgbm_multiclass as lgbm
importlib.reload(lgbm)

from src.LGDcomponent.run.lgbm_multiclass import LgbmMulticlass as LGBM_Multiclass

In [12]:
train_map = {'x_train':X_train, 'y_train':y_train}
val_map = {'x_val':X_val, 'y_val':y_val}

In [13]:
train = LGBM_Multiclass(train_map = train_map, val_map = val_map, config_path = train_cfPath,test_path = test_cfPath)

In [14]:
train.run()

[I 2026-07-03 00:11:59,193] A new study created in memory with name: no-name-b6c1c89c-5fbc-4cd8-9087-9daa08e71a04


[LGDDiscretizer] n_bins ajusté : 8 demandés →  7 bins effectifs (doublons dans la distribution).


[I 2026-07-03 00:12:01,116] Trial 0 finished with value: -0.269477642997252 and parameters: {'max_depth': 6, 'num_leaves': 20, 'min_child_samples': 25, 'reg_lambda': 2.6045177062934255, 'reg_alpha': 0.7404728239870845, 'subsample': 0.6920266063859359, 'colsample_bytree': 0.9361277636326548, 'learning_rate': 0.021074126321868534, 'n_estimators': 175}. Best is trial 0 with value: -0.269477642997252.


RMSE: 0.2695 | Dxy: 0.2967 | ECE: 0.0221


[I 2026-07-03 00:12:02,319] Trial 1 finished with value: -0.2687213676281103 and parameters: {'max_depth': 5, 'num_leaves': 32, 'min_child_samples': 29, 'reg_lambda': 1.2482048020599827, 'reg_alpha': 0.27418621906687557, 'subsample': 0.91142895126439, 'colsample_bytree': 0.9976538673252694, 'learning_rate': 0.04804527471708815, 'n_estimators': 500}. Best is trial 0 with value: -0.269477642997252.


...

RMSE: 0.2679 | Dxy: 0.2999 | ECE: 0.0140


[I 2026-07-03 00:12:42,629] Trial 18 finished with value: -0.27006121771753294 and parameters: {'max_depth': 3, 'num_leaves': 26, 'min_child_samples': 34, 'reg_lambda': 8.246067032461928, 'reg_alpha': 0.646610876875327, 'subsample': 0.9182415724485178, 'colsample_bytree': 0.7706235346747657, 'learning_rate': 0.018462074708191233, 'n_estimators': 475}. Best is trial 16 with value: -0.27387071134622265.


RMSE: 0.2701 | Dxy: 0.2899 | ECE: 0.0169


[I 2026-07-03 00:12:44,366] Trial 19 finished with value: -0.26897399722697557 and parameters: {'max_depth': 3, 'num_leaves': 41, 'min_child_samples': 45, 'reg_lambda': 5.380252805812413, 'reg_alpha': 0.9030809027303509, 'subsample': 0.8519680678364765, 'colsample_bytree': 0.6978204716114585, 'learning_rate': 0.034589094649299656, 'n_estimators': 341}. Best is trial 16 with value: -0.27387071134622265.


RMSE: 0.2690 | Dxy: 0.2955 | ECE: 0.0163
🏃 View run LightGBM LGD Multiclass Train at: http://localhost:5000/#/experiments/3/runs/c941d55209d1427ca872ad2c7e146f73
🧪 View experiment at: http://localhost:5000/#/experiments/3


test

In [15]:
bin_edges = train.discretizer.bin_edges_

In [16]:
bin_edges

array([0.        , 0.12463379, 0.23087378, 0.32667991, 0.42833108,
       0.53819802, 0.72005435, 1.        ])

In [17]:
test_map ={'x_test':X_test, 'y_test':y_test}

In [18]:
test = LGBM_Multiclass(test_map=test_map, config_path = test_cfPath)

In [20]:
test.run()

🏃 View run LightGBM LGD Multiclass Test at: http://localhost:5000/#/experiments/3/runs/5603a911c1e844d885ae922e209689ac
🧪 View experiment at: http://localhost:5000/#/experiments/3


In [20]:
test_discretizer = test.discretizer

In [21]:
test_discretizer.bin_edges_

array([0.        , 0.12463379, 0.23087378, 0.32667991, 0.42833108,
       0.53819802, 0.72005435, 1.        ])

# compute lgd

In [ ]:
#midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2
# [0.0623, 0.1778, 0.2788, 0.3775, 0.4832, 0.6291, 0.8600]

#y_predict = sum(proba[0] * midpoints)
# = 0.1673*0.0623 + 0.0426*0.1778 + 0.0483*0.2788 + 0.0613*0.3775
#   + 0.0533*0.4832 + 0.0908*0.6291 + 0.5363*0.8600
# ≈ 0.587

# Test inference

In [27]:
import src.LGDcomponent.LgdPrediction as lgd
importlib.reload(lgd)
from src.LGDcomponent.LgdPrediction import LGDPrediction

In [28]:
mlflow_config ='/Users/macbookpro/platform/Backend/configs/Lgd_mlFlow_config.yaml'
model_config = '/Users/macbookpro/platform/Backend/configs/Lgd_model_config.yaml'

In [29]:
inference = LGDPrediction(hist=loan_hist,orig=loan_orig,mlflow_config=mlflow_config, model_config=model_config)

Copy DataFrame     : 0.0s
Cast DPD           : 0.0s
Groupby            : 0.0s
Colonnes de travail: 0.0s


In [30]:
discretize = inference.discretizer

In [31]:
type(discretize)

pipelines.Features.Lgd_discretizer.LGDDiscretizer

In [32]:
inference.apply()

[np.float64(0.4757785148901206)]

choisir des profile, et stocker toute leur performance et origination dans le warehouse, pour le Pd, on construira leur windows .

In [ ]:
# window et sans window ( preselection des profile sans window) dans le warehouse,